In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Fixed two-pilot temporal attack evaluation

Run all. The first cell asks you to authorize the Drive mount. This notebook reuses the specified prior two-pilot run and never regenerates a missing main arm.


In [ ]:
# Load the ordinary companion source and initialize the fixed plan before model setup.
import datetime, importlib, json, os, shutil, signal, subprocess, sys, time, traceback, urllib.request, uuid, zipfile
from pathlib import Path

SOURCE_REPOSITORY = "https://github.com/RICHAAARC/SC-SSTW"
SOURCE_REF = "dev/paper-results-v1"
COMPANION_ZIP_URL = SOURCE_REPOSITORY + "/raw/refs/heads/" + SOURCE_REF + "/notebooks/paper_results_v1_companion.zip"
PILOTS = ("pilot_01", "pilot_02")
PHASES = (
    "import-main", "baseline-embed-videoseal", "baseline-embed-rivagan",
    "baseline-codec", "attack-media", "receiver-clock", "receiver-read",
    "baseline-extract-videoseal", "baseline-extract-rivagan", "quality",
)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ID = stamp + "-" + uuid.uuid4().hex[:8]
OUTPUT_ROOT = Path("/content/drive/MyDrive/Video-WM/Paper-Results-V1-Temporal-Attack") / RUN_ID
RUN_OUTPUT = OUTPUT_ROOT / "run_state"
TEMP_ROOT = Path("/content") / ("paper-results-v1-attack-temp-" + RUN_ID)
PORTABLE_ROOT = Path("/content/paper-results-v1-source")
CACHE_ROOT = Path("/content/drive/MyDrive/Video-WM/Paper-Results-V1-Cache")
SOURCE_RUN = Path("/content/drive/MyDrive/Video-WM/Paper-Results-V1-Two-Pilot/20261009T123350716100Z-f5e1f880/run_state")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False); CACHE_ROOT.mkdir(parents=True, exist_ok=True)
LOG = OUTPUT_ROOT / "execution.log"; RECEIPTS = OUTPUT_ROOT / "stage_receipts.json"

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True); tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n"); os.replace(tmp, path)
receipts = []
def record(stage, status, **fields):
    row = {"stage": stage, "status": status, "time_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), **fields}
    receipts.append(row); atomic_json(RECEIPTS, receipts); return row
PYTHON = sys.executable
def logged(command, stage, *, cwd=None, env=None, check=True):
    print(f"[{stage}] start; log={LOG}", flush=True); child = None; rc = None
    started = time.monotonic()
    try:
        with LOG.open("a") as stream:
            stream.write("COMMAND " + repr(command) + "\n"); stream.flush()
            child = subprocess.Popen(command, cwd=cwd or PORTABLE_ROOT, env=env, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
            rc = child.wait()
        record(stage, "COMPLETE" if rc == 0 else "FAILED", returncode=rc, seconds=time.monotonic()-started, command=command)
        if check and rc: raise RuntimeError(f"{stage} exited {rc}")
        return rc
    except BaseException as exc:
        record(stage, "FAILED", returncode=rc, seconds=time.monotonic()-started, command=command, reason=f"{type(exc).__name__}: {exc}")
        raise
    finally:
        if child is not None and child.poll() is None:
            try: os.killpg(child.pid, signal.SIGTERM); child.wait(timeout=5)
            except BaseException:
                try: os.killpg(child.pid, signal.SIGKILL)
                except BaseException: pass
                try: child.wait(timeout=5)
                except BaseException: pass
        print(f"[{stage}] end rc={rc}", flush=True)

required_source = PORTABLE_ROOT / "experiments/paper_results_v1/attack_cli.py"
source_status = "REUSED_EDITABLE_WORKING_DIRECTORY"
archive = OUTPUT_ROOT / "companion_source.zip"
if not required_source.is_file():
    source_status = "DOWNLOADED_AND_EXTRACTED"
    urllib.request.urlretrieve(COMPANION_ZIP_URL, archive)
    PORTABLE_ROOT.mkdir(parents=True, exist_ok=True)
    source_root_resolved = PORTABLE_ROOT.resolve()
    with zipfile.ZipFile(archive) as zf:
        for item in zf.infolist():
            target = (PORTABLE_ROOT / item.filename).resolve()
            if source_root_resolved not in target.parents and target != source_root_resolved:
                raise RuntimeError("unsafe companion archive member")
        zf.extractall(PORTABLE_ROOT)
if not required_source.is_file():
    raise FileNotFoundError("companion source is missing experiments/paper_results_v1/attack_cli.py")
atomic_json(OUTPUT_ROOT / "companion_source_receipt.json", {
    "status": source_status, "source_repository": SOURCE_REPOSITORY,
    "source_ref": SOURCE_REF, "companion_zip_url": COMPANION_ZIP_URL,
    "source_root": str(PORTABLE_ROOT), "manifest_required": False,
    "digest_required": False, "git_clean_required": False,
})
sys.path.insert(0, str(PORTABLE_ROOT))
CONFIG = OUTPUT_ROOT / "effective_attack_config.json"
cfg = json.loads((PORTABLE_ROOT / "experiments/paper_results_v1/attack_eval.adopted.json").read_text())
cfg["source_run"]["run_dir"] = str(SOURCE_RUN)
VIDEOSEAL_COMMIT = "870ca7fb33578b90f14c602016b6c2788096226e"; RIVAGAN_COMMIT = "efffa72a4ca46d4d5051f6970c96424c2cdab441"
WAN_REVISION = "0fad780a534b6463e45facd96134c9f345acfa5b"; FRAMEWISE_REVISION = "31f26fdeee1355a5c34592e401dd41e45d25a493"
WAN_SNAPSHOT = CACHE_ROOT / "models/Wan2.1-T2V-1.3B-Diffusers" / WAN_REVISION
FRAMEWISE_SNAPSHOT = CACHE_ROOT / "models/sd-vae-ft-mse" / FRAMEWISE_REVISION
VS_ROOT = CACHE_ROOT / "sources" / ("videoseal-" + VIDEOSEAL_COMMIT); VS_CARD = VS_ROOT / "videoseal/cards/videoseal_1.0.yaml"; VS_WEIGHT = CACHE_ROOT / "checkpoints/videoseal/y_256b_img.pth"
RIVA_ROOT = CACHE_ROOT / "sources" / ("rivagan-" + RIVAGAN_COMMIT); RIVA_WEIGHT = CACHE_ROOT / "checkpoints/rivagan/rivagan_32bit_model.pt"
cfg["models"]["wan"]["local_snapshot_path"] = str(WAN_SNAPSHOT)
cfg["models"]["framewise"]["local_snapshot_path"] = str(FRAMEWISE_SNAPSHOT)
cfg["models"]["videoseal"].update(source_root=str(VS_ROOT), card_path=str(VS_CARD), checkpoint_path=str(VS_WEIGHT))
cfg["models"]["rivagan"].update(source_root=str(RIVA_ROOT), checkpoint_path=str(RIVA_WEIGHT))
for model_name, optional_fields in {
    "videoseal": ("card_sha256", "checkpoint_sha256"),
    "rivagan": ("checkpoint_sha256",),
}.items():
    for field in optional_fields:
        cfg["models"][model_name].pop(field, None)
atomic_json(CONFIG, cfg)
ENV = {**os.environ, "PYTHONPATH": str(PORTABLE_ROOT)}
portable_env = ENV; EFFECTIVE_CONFIG = CONFIG
record_stage = record
CLI = [sys.executable, "-u", "-m", "experiments.paper_results_v1.attack_cli", "--config", str(CONFIG)]
logged([*CLI, "--output", str(OUTPUT_ROOT / "fixed_plan.json"), "--phase", "plan"], "fixed-plan", env=ENV, check=True)
logged([*CLI, "--output", str(RUN_OUTPUT), "--temp-root", str(TEMP_ROOT), "--phase", "init"], "initialize", env=ENV, check=True)
print("Initialized", RUN_ID, "with all 10 cases; only pilot_01/pilot_02 will be attempted.")


In [ ]:
# Source/checkpoint preparation is isolated per external method.
VIDEOSEAL_REPO = "https://github.com/facebookresearch/videoseal.git"
VIDEOSEAL_CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/videoseal/y_256b_img.pth"
RIVAGAN_REPO = "https://github.com/DAI-Lab/RivaGAN.git"
RIVAGAN_WEIGHT_COMMIT = "4d9928350509f808b6b57d48d2f958aef811d332"
RIVAGAN_CHECKPOINT_URL = "https://raw.githubusercontent.com/Peachypie98/RivaGAN/4d9928350509f808b6b57d48d2f958aef811d332/model_weight/rivagan_32bit_model.pt"
source_checkout_receipts = {}
def prepare_source(label, url, commit, destination):
    destination = Path(destination); created = not destination.exists()
    if created:
        if logged(["git", "clone", "--filter=blob:none", url, str(destination)], label + "_CLONE", check=False):
            raise RuntimeError(label + " clone failed")
        logged(["git", "-C", str(destination), "fetch", "origin", commit], label + "_FETCH", check=False)
        if logged(["git", "-C", str(destination), "checkout", "--detach", commit], label + "_CHECKOUT", check=False):
            raise RuntimeError(label + " checkout failed")
    if not destination.is_dir():
        raise FileNotFoundError(label + " source directory is unavailable: " + str(destination))
    source_checkout_receipts[label] = {
        "status": "CREATED_FROM_REQUESTED_REF" if created else "REUSED_EXISTING_WORKING_DIRECTORY",
        "source_url": url, "requested_ref": commit, "path": str(destination),
    }
    return destination
def cached_download(label, url, destination):
    destination = Path(destination); destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.is_file():
        temporary = destination.with_suffix(destination.suffix + ".download"); temporary.unlink(missing_ok=True)
        urllib.request.urlretrieve(url, temporary); os.replace(temporary, destination)
    source_checkout_receipts[label] = {"status": "AVAILABLE", "source_url": url, "path": str(destination)}
    return destination
baseline_setup = {}
try:
    prepare_source("VIDEOSEAL_SOURCE", VIDEOSEAL_REPO, VIDEOSEAL_COMMIT, VS_ROOT)
    VS_WEIGHT = cached_download("VIDEOSEAL_CHECKPOINT", VIDEOSEAL_CHECKPOINT_URL, VS_WEIGHT)
    if not VS_CARD.is_file(): raise FileNotFoundError("VideoSeal card is unavailable: " + str(VS_CARD))
    baseline_setup["videoseal"] = {"status": "READY", "source_path": str(VS_ROOT), "card_path": str(VS_CARD), "checkpoint_path": str(VS_WEIGHT)}
except Exception as exc:
    baseline_setup["videoseal"] = {"status": "FAILED", "reason": f"{type(exc).__name__}: {exc}"}
try:
    prepare_source("RIVAGAN_SOURCE", RIVAGAN_REPO, RIVAGAN_COMMIT, RIVA_ROOT)
    RIVA_WEIGHT = cached_download("RIVAGAN_CHECKPOINT", RIVAGAN_CHECKPOINT_URL, RIVA_WEIGHT)
    baseline_setup["rivagan"] = {"status": "READY", "source_path": str(RIVA_ROOT), "checkpoint_path": str(RIVA_WEIGHT), "checkpoint_provenance": "Peachypie98 community 32-bit checkpoint; not DAI-Lab official weights"}
except Exception as exc:
    baseline_setup["rivagan"] = {"status": "FAILED", "reason": f"{type(exc).__name__}: {exc}"}
atomic_json(OUTPUT_ROOT / "baseline_setup_receipts.json", baseline_setup)
## Prepare the recorded main environment and isolated baseline dependencies
from experiments.paper_results_v1.colab_orchestration import (
    ensure_baseline_interpreter,
    prepare_baseline_environment,
)

# The historical non-zero pip check is retained as a diagnostic and is not a blanket hard failure.
MAIN_RECOMMENDED_VERSIONS = {
    "torch": "2.11.0", "torchvision": "0.26.0", "diffusers": "0.39.0", "transformers": "4.57.6",
    "numpy": "2.1.3", "accelerate": "1.15.0", "safetensors": "0.8.0",
    "huggingface-hub": "0.36.2", "tokenizers": "0.22.2",
    "sentencepiece": "0.2.2", "ftfy": "6.3.1",
}
distribution_modules = {
    "torch": "torch", "torchvision": "torchvision", "diffusers": "diffusers",
    "transformers": "transformers", "numpy": "numpy", "accelerate": "accelerate",
    "safetensors": "safetensors", "huggingface-hub": "huggingface_hub",
    "tokenizers": "tokenizers", "sentencepiece": "sentencepiece", "ftfy": "ftfy",
}
import importlib.metadata as package_metadata
version_observation = {}
for distribution, recommended in MAIN_RECOMMENDED_VERSIONS.items():
    try:
        actual = package_metadata.version(distribution)
    except package_metadata.PackageNotFoundError:
        actual = None
    version_observation[distribution] = {
        "recommended_historical_version": recommended,
        "actual_version": actual,
        "status": "INSTALLED" if actual else "NOT_INSTALLED",
    }
atomic_json(OUTPUT_ROOT / "main_environment_version_observation.json", version_observation)
probe = "from diffusers import AutoencoderKL,AutoencoderKLWan,WanPipeline; import torch,sentencepiece,ftfy; print('MAIN_IMPORT_READY')"
main_environment_status = "READY"
try:
    if logged([PYTHON, "-c", probe], "MAIN_DEPENDENCY_PROBE", check=False):
        missing = [
            distribution for distribution, module in distribution_modules.items()
            if importlib.util.find_spec(module) is None
        ]
        logged(
            [
                PYTHON, "-m", "pip", "install",
                *[
                    f"{name}=={MAIN_RECOMMENDED_VERSIONS[name]}"
                    for name in (missing or list(distribution_modules))
                ],
            ],
            "MAIN_IMPORT_FAILURE_REPAIR",
        )
        logged([PYTHON, "-c", probe], "MAIN_DEPENDENCY_REPROBE")
    dependency_check_returncode = logged([PYTHON, "-m", "pip", "check"], "MAIN_PIP_CHECK_DIAGNOSTIC", check=False)
    (OUTPUT_ROOT / "environment_freeze.txt").write_text(subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True), encoding="utf-8")
except Exception as exc:
    main_environment_status = "FAILED"
    dependency_check_returncode = None
    record_stage("MAIN_ENVIRONMENT_PREPARATION", "FAILED", reason=f"{type(exc).__name__}: {exc}")

BASELINE_VERSION_RECOMMENDATIONS = OUTPUT_ROOT / "baseline_version_recommendations.json"
atomic_json(BASELINE_VERSION_RECOMMENDATIONS, {
    "status": "HISTORICAL_RECOMMENDATIONS_NOT_CONSTRAINTS",
    "main_versions": MAIN_RECOMMENDED_VERSIONS,
    "version_differences_are_blocking": False,
})
BASELINE_REPAIR_CONSTRAINTS = OUTPUT_ROOT / "baseline_repair_constraints.txt"
BASELINE_REPAIR_CONSTRAINTS.write_text(
    "".join(f"{name}=={version}\n" for name, version in MAIN_RECOMMENDED_VERSIONS.items()),
    encoding="utf-8",
)
baseline_pythons = {}
baseline_dependencies = {
    "videoseal": [
        "PyWavelets", "av", "calflops", "decord", "einops==0.8.2", "lpips",
        "omegaconf==2.3.0", "antlr4-python3-runtime==4.9.*", "PyYAML>=5.1.0",
        "opencv-python", "pandas==2.2.3", "pycocotools", "pytorch-msssim",
        "scikit-image", "scipy==1.16.3", "setuptools", "tensorboard",
        "timm==0.9.16", "tqdm", "transformers", "safetensors",
    ],
    "rivagan": ["torch-dct==0.1.5", "opencv-python", "pandas==2.2.3"],
}
for method in ("videoseal", "rivagan"):
    venv = Path("/content") / ("paper-results-v1-" + method + "-venv")
    vpython = venv / "bin/python"
    baseline_pythons[method] = str(vpython)
    baseline_setup[method]["environment_python"] = str(vpython)

    def ensure_method_interpreter():
        if baseline_setup[method]["status"] != "READY":
            raise RuntimeError(method + " source/checkpoint preparation failed")
        prepared_python, receipt = ensure_baseline_interpreter(
            method=method,
            venv=venv,
            main_python=PYTHON,
            run=logged,
        )
        if prepared_python != str(vpython):
            raise RuntimeError(method + " dedicated interpreter path changed")
        baseline_setup[method]["interpreter_preparation"] = receipt
        baseline_setup[method].setdefault("interpreter_preparation_history", []).append(receipt)
        return prepared_python

    def install_baseline_dependencies():
        # Resolve only after the actual entry import fails. The repair then
        # constrains the known historical main stack; those versions are
        # never a prerequisite when the existing imports already work.
        # The obsolete official RivaGAN requirements file is never installed.
        logged([
            PYTHON, "-m", "pip", "--python", str(vpython), "install",
            "--constraint", str(BASELINE_REPAIR_CONSTRAINTS),
            *baseline_dependencies[method],
        ], method.upper() + "_ISOLATED_DEPENDENCIES")
        baseline_setup[method]["environment_freeze_path"] = str(OUTPUT_ROOT / (method + "_environment_freeze.txt"))
        try:
            freeze = subprocess.check_output(
                [PYTHON, "-m", "pip", "--python", str(vpython), "freeze"],
                text=True,
            )
            (OUTPUT_ROOT / (method + "_environment_freeze.txt")).write_text(
                freeze, encoding="utf-8",
            )
            baseline_setup[method]["environment_freeze_status"] = "RECORDED"
        except Exception as exc:
            baseline_setup[method]["environment_freeze_status"] = "OBSERVATION_UNAVAILABLE"
            baseline_setup[method]["environment_freeze_reason"] = f"{type(exc).__name__}: {exc}"

    def probe_baseline_entry():
        if method == "videoseal":
            probe_code = (
                "import sys; sys.path.insert(0," + repr(str(VS_ROOT)) + "); "
                "from omegaconf import OmegaConf; "
                "from videoseal.utils.cfg import setup_model; "
                "print('VIDEOSEAL_IMPORT_READY_NO_MODEL_LOADED')"
            )
        else:
            probe_code = (
                "import sys; sys.path[:0]=" + repr([str(PORTABLE_ROOT), str(RIVA_ROOT)]) + "; "
                "from experiments.paper_results_v1.real_backends import _install_rivagan_pickle_classes; "
                "_install_rivagan_pickle_classes(); "
                "print('RIVAGAN_IMPORT_READY_NO_CHECKPOINT_LOADED')"
            )
        logged(
            [str(vpython), "-c", probe_code],
            method.upper() + "_ENTRY_IMPORT_PROBE", env=portable_env,
        )

    try:
        ensure_method_interpreter()
    except Exception as exc:
        baseline_setup[method].update(
            interpreter_preparation={
                "status": "FAILED",
                "reason": f"{type(exc).__name__}: {exc}",
                "ensurepip_used": False,
            },
            dependency_install_status="BLOCKED_INTERPRETER_PREPARATION_FAILED",
            entry_import_probe_status="BLOCKED_INTERPRETER_PREPARATION_FAILED",
            model_compatibility_status="NOT_VALIDATED_REQUIRES_REAL_EMBED_EXTRACT",
        )
        continue
    baseline_setup[method].update(prepare_baseline_environment(
        install=install_baseline_dependencies,
        probe=probe_baseline_entry,
    ))
atomic_json(OUTPUT_ROOT / "baseline_setup_receipts.json", baseline_setup)

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token: os.environ["HF_TOKEN"] = hf_token
except Exception:
    hf_token = None
try:
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="Wan-AI/Wan2.1-T2V-1.3B-Diffusers", revision=WAN_REVISION,
        local_dir=str(WAN_SNAPSHOT),
        allow_patterns=["model_index.json", "vae/*"],
    )
    snapshot_download(
        repo_id="stabilityai/sd-vae-ft-mse", revision=FRAMEWISE_REVISION,
        local_dir=str(FRAMEWISE_SNAPSHOT),
        allow_patterns=["config.json", "diffusion_pytorch_model.safetensors"],
    )
    model_snapshot_status = "READY"
except Exception as exc:
    model_snapshot_status = "FAILED"
    record_stage("MODEL_SNAPSHOT_PREPARATION", "FAILED", reason=f"{type(exc).__name__}: {exc}")
finally:
    hf_token = None
atomic_json(OUTPUT_ROOT / "environment_setup_receipt.json", {
    "main_environment_status": main_environment_status,
    "main_pip_check_returncode_nonfatal_diagnostic": dependency_check_returncode,
    "model_snapshot_status": model_snapshot_status,
    "wan_revision": WAN_REVISION, "framewise_revision": FRAMEWISE_REVISION,
    "baseline_version_recommendations_path": str(BASELINE_VERSION_RECOMMENDATIONS),
    "baseline_repair_constraints_path": str(BASELINE_REPAIR_CONSTRAINTS),
    "baseline_environments": baseline_setup,
    "note": "Model/backend compatibility is established only by later phase receipts, not by preparation status.",
})
print("Environment:", main_environment_status, "models:", model_snapshot_status, "pip check diagnostic:", dependency_check_returncode)
# Attack-quality imports are checked independently of the core stack.
quality_environment_status = "READY"
try:
    if logged([PYTHON, "-c", "import lpips,cv2; print('ATTACK_QUALITY_IMPORT_READY')"], "ATTACK_QUALITY_IMPORT_PROBE", check=False):
        logged([PYTHON, "-m", "pip", "install", "lpips", "opencv-python-headless"], "ATTACK_QUALITY_IMPORT_FAILURE_REPAIR")
        logged([PYTHON, "-c", "import lpips,cv2; print('ATTACK_QUALITY_IMPORT_READY')"], "ATTACK_QUALITY_IMPORT_REPROBE")
except Exception as exc:
    quality_environment_status = "FAILED_RETAINED"
    record_stage("ATTACK_QUALITY_ENVIRONMENT", "FAILED", reason=f"{type(exc).__name__}: {exc}")
VS_PYTHON = baseline_pythons["videoseal"]
RV_PYTHON = baseline_pythons["rivagan"]


In [ ]:
# Fixed stage order. A failed baseline stage is retained and does not stop independent main stages.
attempts = []
interpreters = {"baseline-embed-videoseal": VS_PYTHON, "baseline-extract-videoseal": VS_PYTHON, "baseline-embed-rivagan": RV_PYTHON, "baseline-extract-rivagan": RV_PYTHON}
interrupted = None
try:
    for case_id in PILOTS:
        for phase in PHASES:
            python = interpreters.get(phase, sys.executable)
            command = [python, "-u", "-m", "experiments.paper_results_v1.attack_cli", "--config", str(CONFIG), "--output", str(RUN_OUTPUT), "--temp-root", str(TEMP_ROOT), "--phase", phase, "--case-id", case_id]
            try:
                rc = logged(command, case_id + ":" + phase, env=ENV, check=False)
                attempts.append({"case_id": case_id, "phase": phase, "returncode": rc})
                atomic_json(OUTPUT_ROOT / "attempts.json", attempts)
                if rc:
                    logged([sys.executable, "-m", "experiments.paper_results_v1.attack_cli", "--config", str(CONFIG), "--output", str(RUN_OUTPUT), "--phase", "record-failure", "--failed-phase", phase, "--case-id", case_id, "--reason", "subprocess exited " + str(rc)], case_id + ":retain-failure", env=ENV)
            except KeyboardInterrupt as exc:
                interrupted = exc; attempts.append({"case_id": case_id, "phase": phase, "status": "INTERRUPTED"}); atomic_json(OUTPUT_ROOT / "attempts.json", attempts)
                try: logged([sys.executable, "-m", "experiments.paper_results_v1.attack_cli", "--config", str(CONFIG), "--output", str(RUN_OUTPUT), "--phase", "record-failure", "--failed-phase", phase, "--case-id", case_id, "--reason", "KeyboardInterrupt"], case_id + ":retain-interrupt", env=ENV)
                except BaseException: pass
                raise
            except BaseException as exc:
                attempts.append({"case_id": case_id, "phase": phase, "status": "FAILED_RETAINED", "reason": f"{type(exc).__name__}: {exc}"}); atomic_json(OUTPUT_ROOT / "attempts.json", attempts)
                try: logged([sys.executable, "-m", "experiments.paper_results_v1.attack_cli", "--config", str(CONFIG), "--output", str(RUN_OUTPUT), "--phase", "record-failure", "--failed-phase", phase, "--case-id", case_id, "--reason", repr(exc)], case_id + ":retain-launch-failure", env=ENV)
                except BaseException: pass
finally:
    try: logged([sys.executable, "-m", "experiments.paper_results_v1.attack_cli", "--config", str(CONFIG), "--output", str(RUN_OUTPUT), "--phase", "evaluate"], "final-evaluate", env=ENV)
    except BaseException as exc: record("final-evaluate", "FAILED", reason=f"{type(exc).__name__}: {exc}")
    summary = {"run_id": RUN_ID, "scope": {"attempted": list(PILOTS), "confirmation": "NOT_EXECUTED_BY_NOTEBOOK"}, "run_output": str(RUN_OUTPUT), "report": str(RUN_OUTPUT / "evaluation_report.json"), "interrupted": interrupted is not None}
    atomic_json(OUTPUT_ROOT / "handoff_summary.json", summary)
    print(json.dumps(summary, indent=2))
if interrupted is not None: raise interrupted


In [ ]:
# Return this unique Drive directory. Edited RGB and decoded readback stayed in /content temporary storage.
print("Return directory:", OUTPUT_ROOT)
print("Required: handoff_summary.json, execution.log, stage_receipts.json, attempts.json, fixed_plan.json, run_state/attack_run_state.json, run_state/evaluation_report.json, CSV files, records/, and media/.")
print("The eight confirmation cases remain NOT_EXECUTED_BY_NOTEBOOK and are not pilot failures.")
